# Publicar prévias de amostras no GitHub Pages

**IMPORTANTE:** a pasta indicada deve conter apenas prévias de polígonos da classe escolhida. Um patch inteiro 64×64 com vários objetos **não equivale** a um polígono rotulado. Use esta rotina somente quando separar imagens individuais por classe. Para não confundir classes, a execução começa em modo prévia (`PUBLICAR=False`). Os GeoTIFFs e dados originais continuam no Drive. Configure `GITHUB_TOKEN` nos segredos do Colab.

In [ ]:
# Publicação de prévias de amostras para o site GitHub Pages
# O Drive continua sendo a fonte dos arquivos GeoTIFF, XML e Shapefile.
import base64, json, re
from pathlib import Path
import requests
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
REPOSITORIO='Carine-data-analyst/cbers4a-laboratorio'
PASTA_PREVIAS='/content/drive/MyDrive/CBERS4A/CVAT_teste' # ajuste para a pasta das prévias PNG definitivas
CLASSE='Vegetação arbórea'  # opções: 'Vegetação arbórea' ou 'Outros'
IMAGEM_ORIGEM='Area1.tif'
CONJUNTO=''  # 'treino', 'validacao', 'teste' OU vazio, se não definido
PUBLICAR=False
MAX_ARQUIVOS=20   # publicar em pequenos lotes após revisar
MAX_BYTES=2*1024*1024
DESTINOS={'Vegetação arbórea':'amostras/vegetacao_arborea','Outros':'amostras/outros'}
if CLASSE not in DESTINOS: raise ValueError('Classe não permitida neste portfólio.')
pasta=Path(PASTA_PREVIAS)
if not pasta.is_dir():raise FileNotFoundError(pasta)
files=sorted([p for p in pasta.iterdir() if p.is_file() and p.suffix.lower()=='.png' and p.stat().st_size<=MAX_BYTES])[:MAX_ARQUIVOS]
print('ATENÇÃO: todas as imagens selecionadas receberão a classe:',CLASSE)
print('Não classifique patches mistos como um único polígono apenas por estarem nesta pasta.')
for p in files:print(' -',p.name)
if not files:raise RuntimeError('Nenhum PNG elegível nesta pasta.')
if not PUBLICAR:
 print('MODO PRÉVIA. Separe os PNGs por classe e confirme que são amostras anotadas antes de publicar. Mude PUBLICAR=True para iniciar.')
else:
 token=userdata.get('GITHUB_TOKEN')
 if not token:raise RuntimeError('Configure GITHUB_TOKEN nos segredos do Colab.')
 s=requests.Session();s.headers.update({'Authorization':'Bearer '+token,'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
 api='https://api.github.com/repos/'+REPOSITORIO+'/contents/'
 def get(path):
  r=s.get(api+path,params={'ref':'main'},timeout=30)
  if r.status_code==404:return None,None
  r.raise_for_status();d=r.json()
  return d['sha'],base64.b64decode(d['content']).decode('utf-8', errors='replace')
 def put(path, content):
  sha,_=get(path);data={'message':'Publica prévia de amostra '+path,'branch':'main','content':base64.b64encode(content).decode('ascii')}
  if sha:data['sha']=sha
  r=s.put(api+path,json=data,timeout=45)
  if not r.ok:raise RuntimeError(f'{path}: {r.status_code} {r.text[:200]}')
 _,raw=get('amostras/catalogo.json')
 catalogo=json.loads(raw) if raw else {'versao':1,'contagens_por_classe':{},'amostras':[]}
 ids={x.get('id') for x in catalogo['amostras']}
 selecionadas=[]
 for p in files:
  id_amostra=re.sub(r'[^a-zA-Z0-9_-]','_',p.stem)
  # Mesmo ID não poderá ser repetido, mesmo entre classes
  if id_amostra in ids:
   print('Ignorado (já catalogado):',id_amostra);continue
  destino=f'{DESTINOS[CLASSE]}/{p.name}'
  sha,_=get(destino)
  if sha:
   print('Ignorado (nome de arquivo já existente):',destino);continue
  selecionadas.append((p,id_amostra,destino))
 for p,id_amostra,destino in selecionadas:
  put(destino,p.read_bytes())
  catalogo['amostras'].append({'id':id_amostra,'classe':CLASSE,'imagem_origem':IMAGEM_ORIGEM,'conjunto':CONJUNTO,'descricao':'Prévia de amostra anotada','previa':destino,'arquivo':''})
  print('Publicada:',destino)
 # As contagens por classe de polígonos devem ser verificadas no XML/SHAPEFILE,
 # não inferidas da quantidade de PNGs de patches.
 put('amostras/catalogo.json',(json.dumps(catalogo,ensure_ascii=False,indent=2)+'\n').encode('utf-8'))
 print('Concluído: novas prévias serão exibidas na galeria do site.')
